In [ ]:
# %matplotlib qt
import mne
import os
import time
import numpy as np
import os.path as op
import matplotlib.pyplot as plt

# Function to run over all subjects

In [ ]:
def preprocessEEGWashU(fname_raw, data_path, res_path,
                       clean_ocular = True, clean_cardiac = True, make_report = True):
    '''
    Preprocess EEG data from Wash-U dataset. Remove artifacts from horizontal ocular
    and cardiac activity using ICA.
    '''
    # Setup
    fname_report = fname_raw[:-4] + '.html'
    fname_reconst_raw = fname_raw[:-4] + '_eeg.fif'
    fpath_raw = op.join(data_path, fname_raw)
    fpath_report = op.join(res_path, fname_report)
    fpath_reconst_raw = op.join(data_path, '../preprocessed/', fname_reconst_raw)
    figs = []; captions = []
    
    # Read sample data and instantiate Report
    raw = mne.io.read_raw_cnt(fpath_raw, eog='auto', ecg=['EKG'])
    raw.drop_channels(['M2', 'VEOG']) # Drop 'M2' (ref) and 'VEOG' (blinks already corrected)
    if make_report:
        report = mne.Report(title='Ocular and cardiac artifact correction')
        report.add_raw(raw=raw, title='Raw', psd=False)
    
    # Load data into memory
    raw.load_data()

    # Filtering to remove slow drifts - Since filtering is a linear operator, the
    # ICA solution found from the filtered signal can be applied to the unfiltered
    # signal
    filt_raw_ica = raw.copy().filter(l_freq=0.5, h_freq=None)

    # Fitting the ICA solution
    ica = mne.preprocessing.ICA(n_components=30, max_iter='auto', random_state=97)
    ica.fit(filt_raw_ica)
    
    # Find and remove ICA components correlated to artifacts
    eog_indices = []; ecg_indices = []
    
    if clean_ocular:
        # Start with empty list of excluded ICs
        ica.exclude = []

        # Get evoked ocular activity
        eog_evoked = mne.preprocessing.create_eog_epochs(raw).average()
        eog_evoked.apply_baseline(baseline=(None, -0.2))
        figs.append(eog_evoked.plot_joint())
        captions.append('Evoked EOG')

        # Find which ICs match the EOG pattern
        eog_indices, eog_scores = ica.find_bads_eog(raw)
        ica.exclude = eog_indices

        # Add ICA to Report
        if make_report:
            report.add_ica(ica=ica,
                           title='ICA cleaning of ocular artifacts',
                           inst=raw,
                           eog_evoked=eog_evoked,
                           eog_scores=eog_scores,
                           n_jobs=4 # could be increased
                          )

        # Plot ICs applied to raw data with EOG matches highlighted
        if ica.n_components > 15:
            ica_inds = np.arange(ica.n_components)
            figs.append(ica.plot_sources(raw, show_scrollbars=False, picks=ica_inds[:15]))
            captions.append('EOG: Plot ICs applied to raw data with EOG matches highlighted - Part 1')
            figs.append(ica.plot_sources(raw, show_scrollbars=False, picks=ica_inds[15:]))
            captions.append('EOG: Plot ICs applied to raw data with EOG matches highlighted - Part 2')
        else:
            figs.append(ica.plot_sources(raw, show_scrollbars=False))
            captions.append('EOG: Plot ICs applied to raw data with EOG matches highlighted')

    if clean_cardiac:
        # Start with empty list of excluded ICs
        ica.exclude = []

        # Get evoked cardiac activity
        ecg_evoked = mne.preprocessing.create_ecg_epochs(raw).average()
        ecg_evoked.apply_baseline(baseline=(None, -0.2))
        figs.append(ecg_evoked.plot_joint())
        captions.append('Evoked ECG')

        # Find which ICs match the ECG pattern and select only the first ICA component
        ecg_indices_all, ecg_scores_all = ica.find_bads_ecg(raw)
        ecg_indices = [ecg_indices_all[0]]
        ecg_scores = np.zeros((ica.n_components,))
        ecg_scores[ecg_indices] = ecg_scores_all[ecg_indices]
        ica.exclude = ecg_indices

        # Add ICA to Report
        if make_report:
            report.add_ica(ica=ica,
                           title='ICA cleaning of cardiac artifacts',
                           inst=raw,
                           ecg_evoked=ecg_evoked,
                           ecg_scores=ecg_scores,
                           n_jobs=4 # could be increased
                          )

        # Plot ICs applied to raw data with ECG matches highlighted
        if ica.n_components > 15:
            ica_inds = np.arange(ica.n_components)
            figs.append(ica.plot_sources(raw, show_scrollbars=False, picks=ica_inds[:15]))
            captions.append('ECG: Plot ICs applied to raw data with ECG matches highlighted - Part 1')
            figs.append(ica.plot_sources(raw, show_scrollbars=False, picks=ica_inds[15:]))
            captions.append('ECG: Plot ICs applied to raw data with ECG matches highlighted - Part 2')
        else:
            figs.append(ica.plot_sources(raw, show_scrollbars=False))
            captions.append('ECG: Plot ICs applied to raw data with ECG matches highlighted')

    # Combine ICs with ocular and cardiac artifacts
    ica.exclude = eog_indices + ecg_indices
    
    # Reconstruction of raw data after ICA exclusion
    reconst_raw = raw.copy()
    ica.apply(reconst_raw)

    # Visualize repair
    figs.append(raw.plot(n_channels=len(raw), show_scrollbars=False))
    captions.append('Original raw')
    figs.append(reconst_raw.plot(n_channels=len(raw), show_scrollbars=False))
    captions.append('Reconstructed raw')
    
    # Export preprocessed data to a new .fif file
    reconst_raw.save(fpath_reconst_raw, overwrite=True)
    
    # Save final report
    if make_report:
        for fig, caption in zip(figs, captions):
            report.add_figure(fig=fig, title=caption)
        report.save(fpath_report, overwrite=True, open_browser=False)
        plt.close('all')
        
    return reconst_raw

# Given

In [ ]:
data_path = op.expanduser('~/data/wash-u/REST18_20_MZ_Best/')
res_path = op.expanduser('~/research/results/wash-u/reports/')

# Use preprocessing function over all subjects

In [ ]:
# Start clocking execution
start_time = time.time()

fnames = [f for f in os.listdir(data_path) if f[-4:] == '.cnt']
for i_file, fname_raw in enumerate(fnames):
    print('Preprocessing %s (file# %03d)' % (fname_raw, i_file))
    _ = preprocessEEGWashU(fname_raw, data_path, res_path)
    
# End clocking execution and display result
execution_time = (time.time() - start_time)
print('Execution time in seconds:', execution_time)

# Run each preprocessing step like a script

Read sample data and instantiate Report

In [ ]:
if make_report:
    report = mne.Report(title='Ocular and cardiac artifact correction')
figs = []; captions = []
raw = mne.io.read_raw_cnt(fpath_raw, eog='auto', ecg=['EKG'])
raw.drop_channels(['M2', 'VEOG'])
# raw.info

if make_report:
    report.add_raw(raw=raw, title='Raw', psd=False)

Use ICA to remove ocular and cardiac artifacts

In [ ]:
# Load data into memory
raw.load_data()

# Filtering to remove slow drifts - Since filtering is a linear operator, the
# ICA solution found from the filtered signal can be applied to the unfiltered
# signal
filt_raw_ica = raw.copy().filter(l_freq=0.5, h_freq=None)

# Fitting the ICA solution
ica = mne.preprocessing.ICA(n_components=5, max_iter='auto', random_state=97)
ica.fit(filt_raw_ica)
# ica

eog_indices = []; ecg_indices = []

if clean_ocular:
    # Start with empty list of excluded ICs
    ica.exclude = []
    
    # Get evoked ocular activity
    eog_evoked = mne.preprocessing.create_eog_epochs(raw).average()
    eog_evoked.apply_baseline(baseline=(None, -0.2))
    figs.append(eog_evoked.plot_joint())
    captions.append('Evoked EOG')
    
    # Find which ICs match the EOG pattern
    eog_indices, eog_scores = ica.find_bads_eog(raw)
    ica.exclude = eog_indices
    
    # Add ICA to Report
    if make_report:
        report.add_ica(ica=ica,
                       title='ICA cleaning of ocular artifacts',
                       inst=raw,
                       eog_evoked=eog_evoked,
                       eog_scores=eog_scores,
                       n_jobs=4 # could be increased
                      )

    # Plot ICs applied to raw data with EOG matches highlighted
    if ica.n_components > 15:
        ica_inds = np.arange(ica.n_components)
        figs.append(ica.plot_sources(raw, show_scrollbars=False, picks=ica_inds[:15]))
        captions.append('EOG: Plot ICs applied to raw data with EOG matches highlighted - Part 1')
        figs.append(ica.plot_sources(raw, show_scrollbars=False, picks=ica_inds[15:]))
        captions.append('EOG: Plot ICs applied to raw data with EOG matches highlighted - Part 2')

if clean_cardiac:
    # Start with empty list of excluded ICs
    ica.exclude = []
    
    # Get evoked cardiac activity
    ecg_evoked = mne.preprocessing.create_ecg_epochs(raw).average()
    ecg_evoked.apply_baseline(baseline=(None, -0.2))
    figs.append(ecg_evoked.plot_joint())
    captions.append('Evoked ECG')
    
    # Find which ICs match the ECG pattern and select only the first ICA component
    ecg_indices_all, ecg_scores_all = ica.find_bads_ecg(raw)
    ecg_indices = [ecg_indices_all[0]]
    ecg_scores = np.zeros((ica.n_components,))
    ecg_scores[ecg_indices] = ecg_scores_all[ecg_indices]
    ica.exclude = ecg_indices
    
    # Add ICA to Report
    if make_report:
        report.add_ica(ica=ica,
                       title='ICA cleaning of cardiac artifacts',
                       inst=raw,
                       ecg_evoked=ecg_evoked,
                       ecg_scores=ecg_scores,
                       n_jobs=4 # could be increased
                      )

    # Plot ICs applied to raw data with ECG matches highlighted
    if ica.n_components > 15:
        ica_inds = np.arange(ica.n_components)
        figs.append(ica.plot_sources(raw, show_scrollbars=False, picks=ica_inds[:15]))
        captions.append('ECG: Plot ICs applied to raw data with ECG matches highlighted - Part 1')
        figs.append(ica.plot_sources(raw, show_scrollbars=False, picks=ica_inds[15:]))
        captions.append('ECG: Plot ICs applied to raw data with ECG matches highlighted - Part 2')
    
# Combine ICs with ocular and cardiac artifacts
ica.exclude = eog_indices + ecg_indices

# Reconstruction of raw data after ICA exclusion
reconst_raw = raw.copy()
ica.apply(reconst_raw)

# Visualize repair
figs.append(raw.plot(n_channels=len(raw), show_scrollbars=False))
captions.append('Original raw')
figs.append(reconst_raw.plot(n_channels=len(raw), show_scrollbars=False))
captions.append('Reconstructed raw')

# Export preprocessed data to a new .fif file
fname_reconst_raw = fname_raw[:-4] + '_eeg.fif'
fpath_reconst_raw = op.join(data_path, '../preprocessed/', fname_reconst_raw)
reconst_raw.save(fpath_reconst_raw, overwrite=True)

# Save final report
if make_report:
    for fig, caption in zip(figs, captions):
        report.add_figure(fig=fig, title=caption)
    report.save(fpath_report, overwrite=True)
    plt.close('all')